In [2]:
import pandas as pd

If BlinkIT adds a new product with these features… what sales might happen?
New product:
Item Type = Snack Foods
Item Visibility = 0.35
Item Weight = 12
Rating = 4.5
Outlet Type = Supermarket Type3
Outlet Size = Medium
Outlet Location Type = Tier 2
Outlet Establishment Year = 2015
Item Fat Content = Regular

In [14]:
new_product = pd.DataFrame([{
    "Item Fat Content": "Regular",
    "Item Type": "Snack Foods",
    "Outlet Establishment Year": 2015,
    "Outlet Identifier": "OUT017",
    "Outlet Location Type": "Tier 2",
    "Outlet Size": "Medium",
    "Outlet Type": "Supermarket Type3",
    "Item Visibility": 0.10,
    "Item Weight": 12,
    "Rating": 4.5
}])

In [16]:
new_product = new_product.drop(columns=["Outlet Identifier"]) ## because earlier we dropped Item identifier

In [17]:
new_product_encoded = pd.get_dummies(new_product, drop_first=True) ##because our previous Dataformat was on encoding format

In [18]:
## - SALES PREDICTION (REGRESSION)
## - Given details of an item + store, what will its Sales be?
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.ensemble import RandomForestRegressor

df = pd.read_excel(r"D:\blinkit\Blinkit Data\BlinkIT Grocery Data.xlsx")

##print("Shape:", df.shape)
##print("\nColumns:\n", df.columns)
##print("\nMissing values:\n", df.isnull().sum())
y = df["Sales"]
##print(y.head()) ## Sales our Y value Sales
x = df.drop(columns=["Sales", "Item Identifier"]) #here we added all column to X except !
##print(x.columns) ## all without Sales and Item Identifier
x["Item Weight"] = x["Item Weight"].fillna(0) ## replace null with zero we can also remove the same
##print(x.isnull().sum()) ##confirming no missing values

## Convert categorical columns to numbers##
x_encoded = pd.get_dummies(x,drop_first=True)
##print(x_encoded.shape)
##print(x_encoded.columns)
x_encoded_train,x_encoded_test,y_train,y_test = train_test_split(x_encoded,y,test_size=0.2,random_state=42)
model = LinearRegression()
model.fit(x_encoded_train, y_train)
pred = model.predict(x_encoded_test)
##print(pred)
print("Actual:", y_test[:10].values)
print("Predicted:", pred[:10])
r2 = r2_score(y_test, pred) ## y test its the actual result , Pred is what i predicted .
print("R2 Score:", r2) ## - R2 Score: 0.014365362630749834
##R² ≈ 0 → model ≈ average guess.
MSE = mean_squared_error(y_test, pred)
RMSE = np.sqrt(MSE)
print("MSE:", MSE,"RMSE:", RMSE)
##MSE: 3904.498194894506 RMSE: 62.48598398756721- “On average, my prediction is off by ₹62.”
##Compare RMSE to average target value ,Suppose average Sales ≈ ₹140
##RMSE = 62
##62 / 140 ≈ 44%
## Your average prediction error is ~44% of average sales. That’s poor.
##So what is “good”?  ----  Not “close to zero” necessarily — but small relative to Sales.
## - Linear Regression fail - Linear Regression is too simple for the kind of patterns hidden in your BlinkIT data.
##so we willl use random forest which will decrease the RMSE from - 62 to 44
## We will Try random Forest we already have xtrain and test we dont need to write it extra
model_Random_forest = RandomForestRegressor(random_state=42) ##“Create a Random Forest regression model object.” we are
# not training here we are just saying we want to use random forest - we can also use n_estimators=500 to increase the tree
model_Random_forest.fit(x_encoded_train, y_train) ## this is real Learning Stage
pred = model_Random_forest.predict(x_encoded_test)
model_Random_forest.fit(x_encoded_train, y_train)
pred = model_Random_forest.predict(x_encoded_test)
r2_Random_forest = r2_score(y_test, pred)
print("R2 Score(Random Forest):", r2_Random_forest)
MSE_Random_Forest = mean_squared_error(y_test, pred)
RMSE_Random_forest = np.sqrt(MSE_Random_Forest)
print("MSE(Random Forest):", MSE_Random_Forest,"RMSE(Random Forest):", RMSE_Random_forest)

##Your model currently predicts Sales for any given product + outlet setup.

Actual: [183.6634 127.3362 231.1668 188.3898 102.6332  60.4194  56.293   92.5436
 163.1236 165.65  ]
Predicted: [129.32729851 141.85292091 145.44548276 150.57019101 146.30085039
 144.3232918  131.07353668 133.18894844 144.02179654 126.89944329]
R2 Score: 0.014365362630749834
MSE: 3904.498194894506 RMSE: 62.48598398756721
R2 Score(Random Forest): 0.5123575205966691
MSE(Random Forest): 1931.7494621192868 RMSE(Random Forest): 43.951671892196394


In [19]:
new_product_encoded = new_product_encoded.reindex(
    columns=x_encoded.columns,
    fill_value=0
) ##Same columns as training Missing columns = 0

In [9]:
predicted_sales = model_Random_forest.predict(new_product_encoded) ##Predict using Random Forest

In [11]:
print("Predicted Sales for new product:", predicted_sales[0])
##“If I launch this product under these conditions, what sales should I expect?”

Predicted Sales for new product: 127.68811599999998


In [12]:
##Feature Importance ##What truly drives Sales?
importance = model_Random_forest.feature_importances_
feature_names = x_encoded.columns
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)
print(feature_importance_df.head(10))

                            Feature  Importance
1                   Item Visibility    0.321889
2                       Item Weight    0.264617
3                            Rating    0.077588
0         Outlet Establishment Year    0.028209
11                  Item Type_Dairy    0.021563
13  Item Type_Fruits and Vegetables    0.021539
5          Item Fat Content_Regular    0.021345
12           Item Type_Frozen Foods    0.020843
20            Item Type_Snack Foods    0.020697
4          Item Fat Content_Low Fat    0.019472


In [20]:
## if I  change the visibility from 0.35 → 0.10
predicted_sales = model_Random_forest.predict(new_product_encoded)
print("Predicted Sales for new product:", predicted_sales[0])

Predicted Sales for new product: 105.45176799999999


Scenario A — Higher Visibility
Visibility = 0.35
Predicted Sales ≈ ₹127.69

Scenario B — Lower Visibility
Visibility = 0.10
Predicted Sales ≈ ₹105.45

Difference  = ₹127.69 - ₹105.45 = ₹22.24
For this same product:
Lowering visibility from 0.35 → 0.10 may reduce expected sales by ~₹22.
“Improving shelf visibility could meaningfully increase expected sales for similar products.”